# 📘 RubikCare API Reference

**الإصدار: 1.0** | **التاريخ: 1 مايو 2026**

---

## 1️⃣ APIs PSP & Dispense

### `POST` api/psp/entry
**الملف:** `Api.Web/Controllers/PharmaCompany/PspController.Entry.cs`
**الوصف:** نقطة دخول المريض. تستقبل كود الدعوة أو تعرض البرامج النشطة.

**Request:**
```json
{ "invitationCode": "INV-ABC123" }
// أرسل "" لسحب البرامج النشطة
```

**Response (حالات متعددة):**
```json
// NEW_ENROLLMENT
{ "status": "NEW_ENROLLMENT", "message": "تم الاشتراك", "newProgram": { "programId": 2, "programName": "...", "tokenCode": "RC-...", "patientId": 7 } }

// ACTIVE_PROGRAMS
{ "status": "ACTIVE_PROGRAMS", "message": "لديك برامج نشطة", "activePrograms": [ { "programId": 2, "programName": "...", "tokenCode": "RC-..." } ] }

// ALREADY_ENROLLED
{ "status": "ALREADY_ENROLLED", "message": "مسجل مسبقاً" }

// CODE_INVALID / CODE_EXPIRED
{ "status": "CODE_INVALID", "message": "كود غير صالح" }
```

---

### `GET` api/psp/patient-details
**الملف:** `Api.Web/Controllers/PharmaCompany/PspController.cs`
**الوصف:** جلب تفاصيل برنامج المريض ورمز الصرف الحالي.

**Request:** Query Params `?patientId=15&participationId=3`

**Response:**
```json
{
    "programId": 2, "programName": "برنامج", "tokenCode": "RC-...", "tokenStatus": "ACTIVE",
    "originalPrice": 95.0, "discountedPrice": 66.5, "medications": [ { "medicationName": "...", "dosage": "..." } ]
}
```
**ملاحظة:** خاصية `tokenStatus` لها 3 حالات: `ACTIVE`, `DISPENSED`, `RECEIVED`.

---

### `POST` api/dispense/confirm
**الملف:** `Api.Web/Controllers/Pharmacy/DispenseController.cs`
**الوصف:** تأكيد صرف الدواء من قبل الصيدلي.

**Request:**
```json
{ "tokenCode": "RC-20260501-8839", "dispensationId": 18, "pharmacyId": 1258 }
```

**Response:**
```json
{ "success": true, "message": "تم صرف الدواء بنجاح", "transactionId": 18, "dispenseDate": "..." }
```

---

### `POST` api/dispense/confirm-receipt
**الملف:** `Api.Web/Controllers/Pharmacy/DispenseController.cs`
**الوصف:** تأكيد استلام المريض للدواء بعد صرفه.

**Request:**
```json
{ "tokenCode": "RC-20260501-8839", "patientId": 15 }
```

**Response:**
```json
{ "success": true, "message": "تم تأكيد استلام الدواء" }
```

---

### `POST` api/rep/invitations/create
**الملف:** `Api.Web/Controllers/Rep/RepController.cs`
**الوصف:** إنشاء دعوة جديدة من قبل المندوب.

**Request:**
```json
{ "programId": 2, "invitedOrganizationId": 1253, "recipientType": "DOCTOR", "customMessage": "دعوة للانضمام" }
```

**Response:**
```json
{ "success": true, "invitationId": 21, "invitationToken": "REP-6MIJ0TD5", "expiryDate": "..." }
```

---

### `GET` api/rep/invitations/by-token
**الملف:** `Api.Web/Controllers/Rep/RepController.cs`
**الوصف:** البحث عن دعوة مندوب باستخدام الكود.

**Request:** Query Param `?token=REP-6MIJ0TD5`

**Response:**
```json
{ "success": true, "invitationId": 21, "programId": 2, "programName": "...", "invitedOrganizationType": "DOCTOR" }
```

---

### `POST` api/rep/invitations/accept
**الملف:** `Api.Web/Controllers/Rep/RepController.cs`
**الوصف:** قبول دعوة مندوب وإنشاء اشتراك في البرنامج.

**Request:**
```json
{ "invitationToken": "REP-6MIJ0TD5", "organizationId": 1253 }
```

**Response:**
```json
{ "success": true, "message": "تم الاشتراك في البرنامج بنجاح", "participationId": 6 }
```

---

## 2️⃣ الإشعارات (Notifications)

### `GET` api/notifications/list
**الملف:** `Api.Web/Controllers/NotificationsController.cs`
**الوصف:** جلب قائمة الإشعارات الخاصة بالمستخدم الحالي.

**Request:** Query Params `?skip=0&take=20`

**Response:**
```json
[
    { "id": 1, "title": "📦 طلب صرف جديد", "message": "المريض أحمد أرسل رمز صرف", "isRead": false, "createdDate": "2026-05-01T14:00:00", "url": "/pharmacy/1258?token=RC-...", "type": "DISPENSE_REQUEST" }
]
```

---

### `POST` api/notifications/mark-read/{id}
**الملف:** `Api.Web/Controllers/NotificationsController.cs`
**الوصف:** تعليم إشعار معين كمقروء.

**Response:**
```json
{ "success": true }
```

---

### `POST` api/notifications/mark-all-read
**الملف:** `Api.Web/Controllers/NotificationsController.cs`
**الوصف:** تعليم جميع الإشعارات كمقروءة.

**Response:**
```json
{ "success": true }
```

---

## 3️⃣ المنظمات (Organizations)

### `GET` api/organizations/pharmacies
**الملف:** `Api.Web/Controllers/Organizations/OrganizationsController.cs`
**الوصف:** البحث عن صيدليات مع فلترة وتقسيم صفحات.

**Request:** Query Params `?areaId=5&searchTerm=الشهاب&page=1&pageSize=10`

**Response:**
```json
{
    "items": [ { "organizationID": 1262, "organizationNameAr": "صيدلية الشفاء", "phone": "0123456", "areaNameAr": "مصر الجديدة" } ],
    "totalCount": 1, "page": 1, "totalPages": 1
}
```

---

### `GET` api/organizations/pharmacy/{id}
**الملف:** `Api.Web/Controllers/Organizations/OrganizationsController.cs`
**الوصف:** جلب تفاصيل صيدلية محددة.

**Response:**
```json
{ "success": true, "data": { "organizationID": 1262, "organizationNameAr": "صيدلية الشفاء", "phone": "0123456" } }
```

---

## 4️⃣ الموقع الجغرافي (GeoLocations)

### `GET` api/cities/by-country/{countryId}
**الملف:** `Api.Web/Controllers/GeoLocations/CitiesController.cs`
**الوصف:** جلب جميع مدن دولة معينة.

**Response:**
```json
[ { "id": 1, "nameAr": "القاهرة", "nameEn": "Cairo" } ]
```

---

### `GET` api/areas/by-city/{cityId}
**الملف:** `Api.Web/Controllers/GeoLocations/AreasController.cs`
**الوصف:** جلب جميع مناطق مدينة معينة.

**Response:**
```json
[ { "areaId": 5, "areaNameAr": "مصر الجديدة" } ]
```

---

## 5️⃣ العيادات والحسابات (Clinic & Auth)

### `GET` api/clinic/my-clinics
**الملف:** `Api.Web/Controllers/ClinicController.cs`
**الوصف:** جلب عيادات المستخدم الحالي.

**Response:**
```json
[ { "organizationId": 1253, "organizationNameAr": "عيادة د. أحمد", "phone": "0123456", "areaName": "مصر الجديدة", "cityName": "القاهرة", "primarySpeciality": "نساء وتوليد" } ]
```

---

### `GET` api/clinic/my-pharmacies
**الملف:** `Api.Web/Controllers/ClinicController.cs`
**الوصف:** جلب صيدليات المستخدم الحالي.

**Response:**
```json
[ { "organizationId": 1262, "organizationNameAr": "صيدلية الشفاء", "phone": "0123456" } ]
```

---

### `GET` api/clinic/{clinicId}/programs
**الملف:** `Api.Web/Controllers/ClinicController.cs`
**الوصف:** جلب برامج الدعم المشترك فيها العيادة.

**Response:**
```json
[ { "programId": 2, "programName": "Infertility Support Program", "companyName": "ديفارت لاب", "status": "ACTIVE", "joinDate": "2026-03-10", "canInvitePatients": true } ]
```

---

### `POST` api/clinic/create
**الملف:** `Api.Web/Controllers/ClinicController.cs`
**الوصف:** إنشاء منشأة جديدة (عيادة أو صيدلية).

**Request:**
```json
{ "organizationNameAr": "عيادة د. أحمد", "organizationTypeId": 4, "phone": "0123456", "areaId": 5, "specialityId": 3 }
```

**Response:**
```json
{ "success": true, "message": "تم إنشاء المنشأة بنجاح", "organizationId": 1253 }
```

---

### `GET` api/user/profile
**الملف:** `Api.Web/Controllers/UserController.cs`
**الوصف:** جلب الملف الشخصي للمستخدم الحالي.

**Response:**
```json
{ "userProfileId": 15, "fullNameAr": "أحمد محمد", "email": "dr@example.com", "roles": ["Doctor"] }
```

---

### `PUT` api/user/profile
**الملف:** `Api.Web/Controllers/UserController.cs`
**الوصف:** تحديث الملف الشخصي.

**Request:**
```json
{ "fullNameAr": "أحمد محمد علي", "phoneNumber": "0111111" }
```

**Response:**
```json
{ "success": true, "message": "تم تحديث الملف الشخصي" }
```

---

### `GET` api/professional/specialities
**الملف:** `Api.Web/Controllers/ProfessionalController.cs`
**الوصف:** جلب التخصصات الطبية المتاحة.

**Response:**
```json
[ { "specialityId": 1, "specialityNameAr": "نساء وتوليد" } ]
```

---

## 6️⃣ أدوية المرضى والمنظمات

### `GET` api/patient/medications/schedules/{userProfileId}
**الملف:** `Api.Web/Controllers/PatientMedicationsController.cs`
**الوصف:** جدول مواعيد الأدوية لمستخدم محدد.

**Request:** Path `{userProfileId}` + Query `?date=2026-05-01`

**Response:**
```json
[ { "id": 1, "medicationName": "اوكسي فري", "dosage": "قرص مرتين يومياً", "scheduledDateTime": "2026-05-01T08:00:00", "scheduleType": "MEDICATION", "status": "Pending" } ]
```

---

### `GET` api/PharmaCompanyMedications/search
**الملف:** `Api.Web/Controllers/PharmaCompany/PharmaCompanyMedicationsController.cs`
**الوصف:** البحث عن الأدوية المتاحة.

**Request:** Query Params `?term=اوكسي&pageSize=10`

**Response:**
```json
{ "data": [ { "pharmaCompanyMedicationID": 1, "medicationNameAr": "اوكسي فري", "isActive": true } ] }
```

---

## 7️⃣ إدارة المنظمات والأنظمة الصحية

### `GET` api/OrganizationMembers/{organizationId}
**الملف:** `Api.Web/Controllers/Organizations/OrganizationMembersController.cs`
**الوصف:** أعضاء منظمة.

### `GET` api/OrganizationJobTitles/{organizationId}
**الملف:** `Api.Web/Controllers/Organizations/OrganizationJobTitlesController.cs`
**الوصف:** المسميات الوظيفية.

### `GET` api/Specialities
**الملف:** `Api.Web/Controllers/Clinical/SpecialitiesController.cs`
**الوصف:** جميع التخصصات الطبية.

### `GET` api/OrganizationTypes
**الملف:** `Api.Web/Controllers/Clinical/OrganizationTypesController.cs`
**الوصف:** أنواع المنظمات (1=منصة، 2=شركة أدوية، 3=صيدلية، 4=عيادة).

---

## 8️⃣ النظام (System)

### `GET` api/health
**الملف:** `Api.Web/Controllers/HealthController.cs`
**الوصف:** فحص صحة النظام للمراقبة.

**Response:**
```json
{ "status": "Healthy", "time": "2026-05-01T17:00:00Z", "server": "SERVER1", "ip": ["192.168.1.41"] }
```

---

### `GET` api/localization/page/{pageCode}?lang=ar
**الملف:** `RubikCare.Web/Controllers/LocalizationController.cs`
**الوصف:** ترجمات صفحة محددة.

**Response:**
```json
{ "TITLE": "برامج دعم المرضى", "SUBMIT": "إرسال" }
```

---

## 9️⃣ خطط الصرف وطلبات الأدوية

### `GET` api/DispensationPlans/program/{programId}
**الملف:** `Api.Web/Controllers/PSP/DispensationPlansController.cs`
**الوصف:** خطط الصرف لبرنامج.

### `GET` api/DispensationPlans/{planId}
**الملف:** `Api.Web/Controllers/PSP/DispensationPlansController.cs`
**الوصف:** تفاصيل خطة صرف.

### `POST` api/DispensationPlans
**الملف:** `Api.Web/Controllers/PSP/DispensationPlansController.cs`
**الوصف:** إنشاء خطة صرف جديدة.

### `PUT` api/DispensationPlans/{planId}
**الملف:** `Api.Web/Controllers/PSP/DispensationPlansController.cs`
**الوصف:** تحديث خطة صرف.

### `DELETE` api/DispensationPlans/{planId}
**الملف:** `Api.Web/Controllers/PSP/DispensationPlansController.cs`
**الوصف:** حذف خطة صرف.

### `POST` api/pharmacy/medication-request
**الملف:** `Api.Web/Controllers/Pharmacy/MedicationRequestController.cs`
**الوصف:** طلب أدوية من صيدلية.

**Request:**
```json
{ "pharmacyId": 1262, "medicationIds": [1, 2], "quantity": 10, "notes": "عاجل" }
```

**Response:**
```json
{ "success": true, "message": "تم إرسال الطلب بنجاح" }
```

### **القسم الجديد: 10️⃣ التواصل مع الطبيب (Messaging)**

```markdown
## 10️⃣ التواصل مع الطبيب (Messaging)

### `GET` api/messaging/conversations/{id}
**الملف:** `Api.Web/Controllers/Messaging/ConversationsController.cs`
**الوصف:** جلب محادثة محددة بواسطة المعرف.

**Request:** Path `{id}` (int)

**Response:**
```json
{
    "conversationId": 5,
    "patientProfileId": 10,
    "patientName": "أحمد علي",
    "clinicId": 3,
    "clinicName": "عيادات الدكتور سمير",
    "lastMessage": "شكراً جزيلاً...",
    "lastMessageDate": "2026-05-05T10:00:00",
    "status": "ACTIVE",
    "unreadCount": 2
}
```

---

### `GET` api/messaging/conversations/my-patient-conversations
**الملف:** `Api.Web/Controllers/Messaging/ConversationsController.cs`
**الوصف:** جلب قائمة المحادثات للمريض الحالي (يتم جلب UserProfileId من التوكن).

**Request:** Query Params `?page=1&pageSize=20`

**Response:**
```json
[
    { 
        "conversationId": 5, 
        "clinicName": "عيادات الدكتور سمير",
        "lastMessage": "شكراً جزيلاً...",
        "unreadCount": 2
    }
]
```

---

### `POST` api/messaging/conversations
**الملف:** `Api.Web/Controllers/Messaging/ConversationsController.cs`
**الوصف:** بدء محادثة جديدة بين مريض وعيادة.

**Request:**
```json
{
    "patientProfileId": 10,
    "clinicId": 3
}
```

**Response:** (نفس نموذج `ConversationResponseDto`)

---

### `GET` api/messaging/messages/conversation/{conversationId}
**الملف:** `Api.Web/Controllers/Messaging/MessagesController.cs`
**الوصف:** جلب رسائل محادثة محددة مع Pagination.

**Request:** Path `{conversationId}` + Query `?page=1&pageSize=50`

**Response:**
```json
{
    "messages": [
        {
            "messageId": 101,
            "senderName": "أحمد علي",
            "senderRole": "PATIENT",
            "content": "السلام عليكم دكتور...",
            "createdAt": "2026-05-05T09:00:00",
            "isRead": true
        }
    ],
    "totalCount": 15,
    "page": 1,
    "pageSize": 50
}
```

---

### `POST` api/messaging/messages
**الملف:** `Api.Web/Controllers/Messaging/MessagesController.cs`
**الوصف:** إرسال رسالة جديدة (نص أو مستند).

**Request:**
```json
{
    "conversationId": 5,
    "senderProfileId": 10,
    "messageType": "TEXT",
    "content": "موعدي القادم يوم الأحد"
}
```

**Response:** (نفس نموذج `MessageResponseDto` مع `DeliveredAt`)

---

### `POST` api/messaging/messages/{messageId}/read-by-patient & /read-by-clinic
**الملف:** `Api.Web/Controllers/Messaging/MessagesController.cs`
**الوصف:** تحديث حالة قراءة الرسالة (للمريض أو للعيادة).

**Request:** Path `{messageId}`

**Response:**
```json
{ "success": true }
```


---

## 📦 أسماء DTOs حسب المجلد

| المجلد | DTOs المهمة |
|--------|------------|
| `Application/DTOs/PSP/` | `PspEntryResponse`, `ActiveProgramDto`, `EnrolledProgramDto`, `PatientProgramDetailsResponse`, `PspProgramDto`, `PspProgramDetailsDto`, `PspProgramMedicationDto`, `MedicationDetail`, `ParticipateDto` |
| `Application/DTOs/Rep/` | `CreateInvitationDto`, `InvitationResponseDto`, `RepProgramDto`, `DoctorSearchDto`, `RecentInvitationDto`, `AcceptRepInvitationRequest` |
| `Application/DTOs/Pharmacy/` | `PharmacyDto`, `PharmacySearchResponse` |
| `Application/DTOs/User/` | `AreaDto`, `UserDto` |
| `Application/DTOs/Locations/` | `CityDto`, `CountryDto`, `GeoAreaDto` |
| `Application/DTOs/Auth/` | `LoginDto`, `RegisterDto`, `UserProfileDto` |
| `Application/DTOs/Organization/` | `UserClinicDto`, `CreateClinicDto`, `ClinicResponseDto`, `ClinicProgramDto` |
| `Application/DTOs/Patient/` | `MedicationScheduleDto` |

---

## 📋 الجدول الكامل (42 API)

| # | Method | Endpoint | القسم |
|---|--------|----------|-------|
| 1 | POST | `api/psp/entry` | PSP |
| 2 | GET | `api/psp/patient-details` | PSP |
| 3 | GET | `api/psp/programs` | PSP |
| 4 | GET | `api/psp/programs/{id}` | PSP |
| 5 | POST | `api/psp/participate` | PSP |
| 6 | POST | `api/psp/create-invitation` | PSP |
| 7 | POST | `api/psp/accept-invitation` | PSP |
| 8 | POST | `api/psp/accept-rep-invitation` | PSP |
| 9 | GET | `api/psp/invitation/by-token` | PSP |
| 10 | POST | `api/dispense/validate-token` | Dispense |
| 11 | POST | `api/dispense/confirm` | Dispense |
| 12 | POST | `api/dispense/confirm-receipt` | Dispense |
| 13 | POST | `api/dispense/send-to-pharmacy` | Dispense |
| 14 | GET | `api/rep/programs` | Rep |
| 15 | POST | `api/rep/invitations/create` | Rep |
| 16 | GET | `api/rep/invitations/by-token` | Rep |
| 17 | POST | `api/rep/invitations/accept` | Rep |
| 18 | GET | `api/rep/dashboard/stats` | Rep |
| 19 | GET | `api/notifications/list` | Notifications |
| 20 | GET | `api/notifications/unread-count` | Notifications |
| 21 | POST | `api/notifications/mark-read/{id}` | Notifications |
| 22 | POST | `api/notifications/mark-all-read` | Notifications |
| 23 | GET | `api/organizations/pharmacies` | Organizations |
| 24 | GET | `api/organizations/pharmacy/{id}` | Organizations |
| 25 | GET | `api/clinic/my-clinics` | Clinic |
| 26 | GET | `api/clinic/my-pharmacies` | Clinic |
| 27 | GET | `api/clinic/{id}/programs` | Clinic |
| 28 | POST | `api/clinic/create` | Clinic |
| 29 | GET | `api/cities/by-country/{id}` | Geo |
| 30 | GET | `api/areas/by-city/{id}` | Geo |
| 31 | GET | `api/user/profile` | User |
| 32 | PUT | `api/user/profile` | User |
| 33 | POST | `api/auth/login` | Auth |
| 34 | POST | `api/auth/register` | Auth |
| 35 | GET | `api/health` | System |
| 36 | GET | `api/localization/page/{pageCode}` | System |
| 37 | GET | `api/DispensationPlans/program/{programId}` | PSP |
| 38 | GET | `api/DispensationPlans/{planId}` | PSP |
| 39 | POST | `api/DispensationPlans` | PSP |
| 40 | PUT | `api/DispensationPlans/{planId}` | PSP |
| 41 | DELETE | `api/DispensationPlans/{planId}` | PSP |
| 42 | POST | `api/pharmacy/medication-request` | Pharmacy |

---

**تم التحديث في: 1 مايو 2026** | **الإصدار: 1.0**


---

## 📄 **1. تحديث `api-reference.ipynb` (وثيقة APIs)**

الهدف: إضافة جميع APIs المراسلة الجديدة وتحديث حالة الإنجاز.

### **تحتاج إلى إضافة هذا القسم الجديد:**

```markdown
## 10️⃣ نظام المراسلة (Messaging) - ✅ تم الإنجاز بالكامل

تم إضافة نظام متكامل للتواصل بين المرضى والعيادات. يعتمد النظام على `BlazorWebView` ويسمح بالمحادثات المباشرة وإدارة الرسائل.

### `POST` api/messaging/conversations
**الملف:** `Api.Web/Controllers/Messaging/ConversationsController.cs`
**الوصف:** بدء محادثة جديدة بين مريض (مستخلص من الـ Token) وعيادة.

**Request:**
```json
{ "clinicId": 1247 }
```
**Response:** `ConversationResponseDto` (معرّف المحادثة، اسم العيادة، الحالة...)

### `GET` api/messaging/conversations/{id}
**الوصف:** جلب تفاصيل محادثة محددة لطرف معين (يتحقق من الصلاحية).

### `GET` api/messaging/messages/conversation/{conversationId}
**الوصف:** جلب رسائل محادثة مع Pagination (يحدد الرسالة المقروءة تلقائياً للمستخدم الحالي).

### `POST` api/messaging/messages
**الملف:** `Api.Web/Controllers/Messaging/MessagesController.cs`
**الوصف:** إرسال رسالة جديدة (نص فقط حاليًا، مع دعم للمرفقات مستقبلاً).

**Request:**
```json
{
  "conversationId": 5,
  "senderProfileId": 10,
  "messageType": "TEXT",
  "content": "نص الرسالة"
}
```
**Response:** `MessageResponseDto` (حالة `Delivered`، معرف الرسالة...)
```

### **وتحتاج إلى تحديث جدول الـ DTOs بإضافة السطر التالي:**
```markdown
| `Application/DTOs/Messaging/` | `ConversationResponseDto`, `MessageResponseDto`, `SendMessageRequestDto`, `MessagesResponseDto` | ✅ جديد |
```

---